# Lab 4: ElasticSearch

Up to this point, we've constructed Information Retrieval pipelines from scratch, which is essential to understand the details of each component.

Yet, within industry settings, there's a necessity for more sophisticated software to conduct efficient searches on a large scale. In this lab we will introduce Elasticsearch (ES), a powerful tool for indexing, searching, and retrieving documents or data based on user queries. In addition, ES has many in-built functions for advanced analytics, and allows for more customisable search.

We will explore some fundemental ES processes in this lab which may help you in Assignment 2.


## Troubleshooting

The Elasticsearch API can sometimes be unstable in Colab and you may run into errors from time to time. Here are some troubleshooting tips:
* Only run the code from Section 1 once. This means that the server and client are only downloaded and installed once, and process duplication is avoided.
* If you create an index which contains zero documents, this is incorrect and this problem is solved by recreating the index using the code provided.
* If your index contains twice as many documents as expected, try to delete the index (code provided below) and recreate.
* When ES server instances are duplicated, you may run into additional errors. To check if you have multiple processes running as daemon use the following code `%%bash ps -ef | grep elasticsearch`. To kill the daemon process you can use   `!kill -9 <id>` where id is the number in the second column related to the daemon process. Failing that, please ask for help from one of the demonstrators.


### 1. Download Elasticsearch server and client

In [ ]:
# The following bash scripts download the elastic search library and install it
# on the google colab instance.

# You need to run these only once when you work on your search engine notebook

# NOTE: If you are working on a large dataset (20k+ docs) you should do this locally
# i.e. in a jupyter notebook. This way you only need to install ES once and
# index your data once

In [ ]:
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!sudo chown -R daemon:daemon elasticsearch-7.9.2/
!shasum -a 512 -c elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512

In [ ]:
# https://stackoverflow.com/questions/68762774/elasticsearchunsupportedproducterror-the-client-noticed-that-the-server-is-no#answer-68918449
!pip install elasticsearch==7.9.1 -q

In [ ]:
# check elasticsearch version in environment
!pip freeze | grep elasticsearch
# If get error:"AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead. Please run next line and follow the system instruction to restart the session.
# !pip install "numpy<2.0"

In [ ]:
# import utility packages
import urllib.request
from bs4 import BeautifulSoup
import re
import time

# let's import ES
from elasticsearch import Elasticsearch

In [ ]:
%%bash --bg
sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch

In [ ]:
%%bash
ps -ef | grep elasticsearch

In [ ]:
# start es server
time.sleep(20) # give the server 20 seconds to start
!curl -X GET "http://localhost:9200"

In [ ]:
es = Elasticsearch("http://localhost:9200")

# Let's test whether we have succesfully started an ES instance and
# imported the python library
if es.ping():
  print('ES instance working')
else:
  print('ES instance not working')

In [ ]:
# Server information
es.info()

### 2. Document Preprocessing

In [ ]:
# The first thing we want to do is to index some data. Let's use our poems from
# the previous lab this time let's get the authors, too:

In [ ]:
def get_boat_poems():
  poems = []
  titles = []
  authors = []
  # get poems in HTML
  url = 'https://discoverpoetry.com/poems/poems-about-ships/'
  contents = urllib.request.urlopen(url).read()
  soup = BeautifulSoup(contents)

  pattern_html = re.compile('<.*?>')

  for poem_html in soup.find_all('article', {'class': 'poem-listing'}):
    # This still contains HTML tags.
    # We need to remove everything between each pair of "<" and ">":
    poem = re.sub(pattern_html, '', str(poem_html))

    # Each document contains "\xa0Full Text" which is not meaningful for our
    # search. It could potentially negatively affect the search ranking since
    # the term "text" is in the english vocabulary. Let's remove it:
    poem = poem.replace('\xa0Full Text', '')

    # In one of the poems, some of the words are emphasised with underscores.
    # "bells_" and "bells" would be classed as different terms by
    # CountVectorizer. This may be useful for searching some dataset,
    # for example, ones that contain programming code. However, in the case of
    # poems we most likely want them removed:
    poem = poem.replace('_', '')

    # Remove for cosmetic reasons:
    poem = poem.replace('►▼', '')

    poems.append(poem)

  # Similarly, we can retrieve the titles in a separate list:
  for title_html in soup.find_all('h3', {'class': 'cat-poem-title'}):
    titles.append(re.sub(pattern_html, '', str(title_html)))

  for author in soup.find_all('div', {'class': 'intro'}):
    authors.append(re.sub(pattern_html, '', str(title_html)).replace('by ', ''))

  return [(poem, title, author) for poem, title, author in  zip(poems, titles, authors)]

In [ ]:
# note that we have now stored the author and title fields as well.
corpus = get_boat_poems()

In [ ]:
# NOTE:
# Compare the two ways below of printing out a poem.
# Pay attention to the "new line" and "carriage return" characters. Fortunatly,
# the vectoriser functions interprets them correctly so we don't need to worry
# about loosing words that happen to go straight after these special characters
# like "\nOld", \nShall, etc.
print('Output from "print(corpus[0][0])":')
print('')
print(corpus[0][0])
print('')
print('What "corpus[0][0]" acutually contains:')
print('')
corpus[0][0]

## 3. Create an ES Index with BM25 Weights

Now let's start indexing some documents.

The key two places to find information on the functionality and what you can do with ES are:
1. the python library documentation: https://elasticsearch-py.readthedocs.io/en/v7.13.2/api.html
2. The general ES documentation: https://www.elastic.co/guide/index.html
  
  2.1 ES Index API: https://www.elastic.co/guide/en/elasticsearch/reference/7.x/docs-index_.html

  2.2 ES Search API: https://www.elastic.co/guide/en/elasticsearch/reference/current/search-search.html

In [ ]:
# Uncomment if you've already created an index. As you'll need to delete to recreate
#es.indices.delete(index_name)

In [ ]:
# mappings are used to define what kind of structure your data has. here explicit mapping is used:
# https://www.elastic.co/guide/en/elasticsearch/reference/current/explicit-mapping.html

# The mapping is used when creating the index through the request body:

request_body = {
    'settings': {
        # sharding is not particularly relevant for us, but you can read about it here: https://www.baeldung.com/java-shards-replicas-elasticsearch
        'number_of_shards': 1,
        'number_of_replicas': 1,

        # which retrieval model to use
        'similarity': {
                'default': {
                    'type': 'BM25',
                    "b": 0,
                    "k1": 10
                }
            }

    },
    # preprocessing
    'mappings': {
          'properties': {
              'title': {'type': 'text'},
              'body': {'type': 'text'},
              'author': {'type': 'text'}
          }
    }
}

index_name = 'poems-about-ships'
try:
  es.indices.get(index_name)
  print('index {} already exists'.format(index_name))
except:
  print('creating index {}'.format(index_name))
  es.indices.create(index_name, body=request_body)

In [ ]:
# now what we want to do is put some data in the index, i.e. index it:
for body, title, author in corpus:
  doc_body = {
      'title': title,
      'body': body,
      'author': author
  }
  es.index(index_name, doc_body)

In [ ]:
# Now let's have a look at our index:
count = es.cat.count(index=index_name,h=['count'])
print(f'We have made and index called {index_name} with {count} documents')

In [ ]:
def index_info(index_name):
  count, deleted, shards, =  es.cat.indices(index=index_name,
                                            h=['docs.count', 'docs.deleted', 'pri'])[:-1].split(' ')
  print(
      """
      #### INDEX INFO #####
      index_name = {}
      doc_count = {}
      shard_count = {}
      deleted_doc_count = {}
      """.format(index_name, count, shards, deleted)
  )


index_info(index_name)

## 4. ES Search

In [ ]:
# now let's try some queries:
# Here the key is the es.search class and the Seach API documentation:
# https://www.elastic.co/guide/en/elasticsearch/reference/current/search-search.html

query_body = {
    'query':{
        'term': {
            'body':  'ships'
        }
    }
}
print('### RESULTS ####')
results = es.search(index=index_name, body=query_body, explain=True)
print('Number of hits: {}'.format(results['hits']['total']['value']))
print('')

for hit in results['hits']['hits']:
  title = hit['_source']['title']
  score = hit['_score']
  print(f'title: "{title}", score: {score}')

print('')
print('More details (in the same order):')
print('')

for hit in results['hits']['hits']:
  explanation = hit['_explanation']
  print(f'more details: {explanation}')

## Task 1:
Create a new index using the Divergence-From-Randomness (DFR).
Use the request body provided below. Search using both DFR and BM25 indexes for a single query, and compare the ranking.

In [ ]:
request_body = {
    'settings': {
        'number_of_shards': 1,
        'number_of_replicas': 1,
        'index': {
            'similarity': {
                'dfr_similarity': {
                    'type': 'DFR',
                    'basic_model': 'g',
                    'after_effect': 'l',
                    'normalization': 'h2',
                    'normalization.h2.c':'3.0'

                }
            }
        }

    },
    'mappings': {
          'properties': {
              'title': {'type': 'text', 'similarity': 'dfr_similarity'},
              'body': {'type': 'text', 'similarity': 'dfr_similarity'},
              'author': {'type': 'text', 'similarity': 'dfr_similarity'}
          }
    }
}

## Task 2:
Have you noticed how a query with a term "ship" ignores the term "ships" and vice versa? Chances are that a user who searches for one would be interested in the other.
Similarly, if they made a typo, e.g. "shpis", the search wouldn't return any
results.
How would you solve these problems?

HINT:
Have a look at the documentation for Fuzzy Queries:
https://www.elastic.co/guide/en/elasticsearch/reference/7.9/query-dsl-fuzzy-query.html

NOTE: By default ES returns only the first 10 hits.

## Task 3:

Investigate parameter tuning in BM25 using the ES indexing settings. Set the BM25 parameters in a way that makes the model highly sensitive to document length. Create a separate index where BM25 is insensitive to document lengths. Compare the search results for a given query when using these two models.